# edasynth quickstart

Reproduces the main results of [github.com/cbrg0/edasynth](https://github.com/cbrg0/edasynth) in the browser.

- **Part 1 (under a minute):** the synthetic map and every check, recomputed from the result tables in the repository.
- **Part 2 (about 10 minutes):** the real-data result, recomputed from the public data.
- **Part 3 (about 2 minutes):** the unit tests.

The full grid (300 scenarios × 50 replicates) takes 1–2 days and is not run here; its 11 300 rows are in `results/grid-metrics-utd.csv`, and Part 1 recomputes the conclusions from them.

**Run all:** menu *Runtime → Run all*.

In [ ]:
# setup: download the repository and install it (about 1 minute)
!git clone -q https://github.com/cbrg0/edasynth.git
%cd edasynth
!pip install -q -e ".[dev]"
import sys
sys.path.insert(0, "src")  # the running notebook does not see a fresh editable install

## Part 1. The synthetic map, recomputed from the raw replicate table

Gain Δ = AUPRC(EDA + acceleration features) − AUPRC(EDA only). *c* is the share of artifacts caused by movement. FB: four coupling features; FA4: four movement-strength features. A scenario is *useful* when the mean gain is at least 0.01 and the lower 95 % bound is above 0.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from edasynth import cli
from edasynth.config import loadConfig
from edasynth.experiment import summarise
from edasynth.figures import f0Summary

cfg = loadConfig(None)
grid = cli.readGrid(Path("results/grid-metrics-utd.csv"), cfg)  # 11 300 replicates, equivalent cells filled in
print(f"{grid.groupby(['c','kappa','deltaMax','piR','classes']).ngroups} scenarios, {len(grid)} replicate rows (11 300 computed, the rest copies of equivalent scenarios)")

by = ["c", "kappa", "deltaMax", "piR", "classes"]
useful = {d: int(summarise(grid, by, d, cfg["analysis"]["decision"])["useful"].sum()) for d in ("dA", "dA4", "dB")}
print("useful scenarios out of 300:", useful)

mix = grid[grid.classes == "C1+C2+C3+C5"]
print("\nmean gain in the mixture of artifact classes, by c:")
print(mix.groupby("c")[["dA", "dA4", "dB"]].mean().round(3))

In [ ]:
summary = pd.read_csv("results/grid-summary-utd.csv")
f0Summary(summary, Path("colab-figures"), cfg["analysis"]["observedDelta"])
display(Image("colab-figures/F0-summary.png", width=640))

### The checks of the generator and the checks on independent data

In [ ]:
v = json.load(open("results/validation-utd.json"))
print("G2 parameter recovery:", "PASS" if v["G2"]["passed"] else "FAIL")
print(f"G3 realism, clean windows: median AUC {v['G3']['auc']:.3f} (limit 0.70)")
print(f"G4 realism, artifact windows: median AUC {v['G4']['auc']:.3f} (limit 0.70)")
print("G5 no spurious gain:", "PASS" if v["G5"]["passed"] else "small losses, no gain (accepted exception)")
print("G6 a built-in link is detected:", "PASS" if v["G6"]["passed"] else "FAIL")

e = json.load(open("results/external-edabe.json"))
print(f"\nEDABE (another device, other experts): artifact windows AUC {e['E1z']['median']:.3f}, clean windows AUC {e['E2z']['median']:.3f}")
print(f"detector trained on synthetic data: AUPRC {e['E3a']['median']:.3f}; trained on real UTD data: {e['E3b']:.3f}")

w = json.load(open("results/external-wesad.json"))
for cell in w["cells"]:
    print(f"WESAD movement, c = {cell['c']}: gain FB {cell['dB_wesad']['mean']:+.3f} (UTD movement {cell['dB_utd']['mean']:+.3f})")

## Part 2. The real-data result (about 10 minutes)

Downloads the PhysioNet recordings and the expert labels of Zhang et al. (2017), checks their SHA-256 checksums, and compares a detector with EDA features only against one that also sees the acceleration (leave one participant out). Expected: AUPRC 0.4486 → 0.4754, Δ +0.0269, 95 % CI −0.0022 to +0.0580 for logistic regression.

In [ ]:
!bash scripts/download-data.sh
!python -m edasynth realdata --stored

## Part 3. Unit tests (about 2 minutes)

In [ ]:
!python -m pytest -q -m "not slow"